# Processing UNC System salary data

Salaries of all employees of the State of North Carolina are public information. Salaries of the UNC System can all be downloaded here:  
https://uncdm.northcarolina.edu/salaries/index.php

The data comes as an excel file and this notebook shows how to process that file into something more palatable for visualization. The specific intention here is to generate a CSV file with the following columns:

- school_abbr
- school_name
- salary
- job_category, One of:
  - administrator,
  - assistant_professor,
  - associate_professor,
  - professor, or
  - lecturer

The raw data, though, includes many more employees with other jobs, though, does not break down faculty vs staff, and doesn't assign faculty into nearly so tidy a list of job categories. So there's a fair amount of data wrangling.

## Imports

In [1]:
import pandas as pd   # Data manipulation library
import re             # Regular expressions for string manipulation

## Read the raw data

In [2]:
data = pd.read_excel('./SystemSalariesOct2025.xlsx')
data = data[data['JOB CATEGORY'].apply(lambda s: not pd.isna(s))]
data = data[data['PRIMARY WORKING TITLE'].apply(lambda s: not pd.isna(s))]
data

,INSTITUTION NAME,LAST NAME,FIRST NAME,INIT,AGE,INITIAL HIRE DATE,JOB CATEGORY,EMPLOYEE ANNUAL BASE SALARY,EMPLOYEE HOME DEPARTMENT,PRIMARY WORKING TITLE
0,ASU,Abbaraju,Pranith,NaN,34,"AUG 01, 2025",Assistant Professor,131500.0,Computer Information Systems,Assistant Professor
1,ASU,Absher,Dianne,S,63,"MAY 23, 2005",Food Prep Worker,37877.0,Bake Shop,Food Service Technician
2,ASU,Abu-Elzait,Sohad,NaN,45,"AUG 01, 2021",Assistant Professor,84020.0,Sustainable Technlgy & Built Envirn,Assistant Professor
3,ASU,Acikgoz,Yalcin,NaN,42,"AUG 26, 2011",Associate Professor,86300.0,Psychology,Associate Professor
4,ASU,Adams,Debbie,S,59,"JUL 31, 2001",Nursing Professional,62775.0,Health Services,Professional Nurse
...,...,...,...,...,...,...,...,...,...,...
49226,WSSU,Young,Cayen,NaN,26,"NOV 01, 2022",General Maintenance Worker,49111.0,Facilities Management,Utilities Plant Operator
49227,WSSU,Zabaleta-Valencia,Minerva,NaN,22,"MAY 12, 2025",Police Officer,54000.0,Police & Public Safety,Police Officer I
49228,WSSU,Zhang,Jinghua,NaN,50,"AUG 13, 2007",Associate Professor,102835.0,Computer Science,Professor
49229,WSSU,Zhang,Lei,NaN,59,"AUG 15, 2005",Associate Professor,90064.0,Chemistry,Professor


## Find and classify faculty

The data has two different fields that we might use to 
- JOB CATEGORY and
- PRIMARY WORKING TITLE

They don't always agree, though and it turns out that JOB CATEGORY is easier to work with and seems to return correct results. There are 8893 PRIMARY WORKING TITLEs and only 969 JOB CATEGORYs.

Thus, our first objective is to identify the various JOB CATEGORYs that correspond to faculty and classify those into one of our canonical job_categorys. That's the purpose of the next block of code.

In [3]:
# Define regular expression patterns to recognize the various
# job_category classes.
prof_pattern = re.compile('prof', re.IGNORECASE)
assist_prof_pattern = re.compile('assis.*prof', re.IGNORECASE)
assoc_prof_pattern = re.compile('assoc.*prof', re.IGNORECASE)
distinguished_pattern = re.compile('Distinguish')
instr_pattern = re.compile('(lectur|instructor)', re.IGNORECASE)
adj_pattern = re.compile('adj', re.IGNORECASE)
admin_pattern1 = re.compile('(dean|provost)', re.IGNORECASE)
admin_pattern2 = re.compile('(Chan|VC)')
bad_admin_pattern = re.compile('ass.*to', re.IGNORECASE)
emer_pattern = re.compile('emer', re.IGNORECASE)
other_pattern = re.compile('chair|director|advisor|fellow|faculty|teach.*ass', re.IGNORECASE)

def normalize_position(position):
    normalized_position = False
    prof_match = prof_pattern.search(position)
    emer_match = emer_pattern.search(position)
    instr_match = instr_pattern.search(position)
    adj_match = adj_pattern.search(position)
    admin_match1 = admin_pattern1.search(position)
    admin_match2 = admin_pattern2.search(position)
    bad_admin_match = bad_admin_pattern.search(position)
    admin_match = (admin_match1 or admin_match2) and not bad_admin_match
    if(admin_match and not bad_admin_match):
        normalized_position = "administrator"
    elif(prof_match and (not adj_match) and (not instr_match) and (not emer_match) and
       (not 'rofessional' in position)):
        if(assist_prof_pattern.search(position)):
            normalized_position = "assistant_professor"
        elif(assoc_prof_pattern.search(position)):
            normalized_position = "associate_professor"
        else:
            normalized_position = "professor"
    elif(instr_match and (not adj_match) and (not 'High' in position)):
        normalized_position = "lecturer"
    elif(adj_match):
        normalized_position = "adjunct"
    # elif(other_pattern.search(position)):
    #     normalized_position = "adjunct"
    return normalized_position

def normalize_slashed_position(slashed_position):
    positions = slashed_position.split("/")
    normalized_position_list = [normalize_position(p) for p in positions]
    if 'administrator' in normalized_position_list:
        normalized_position = 'administrator'
    elif 'professor' in normalized_position_list:
        normalized_position = 'professor'
    elif 'associate_professor' in normalized_position_list:
        normalized_position = 'associate_professor'
    elif 'assistant_professor' in normalized_position_list:
        normalized_position = 'assistant_professor'
    elif 'lecturer' in normalized_position_list:
        normalized_position = 'lecturer'
    elif 'adjunct' in normalized_position_list:
        normalized_position = 'adjunct'
    else:
        normalized_position = False
    return normalized_position

In [4]:
# Normalize JOB CATEGORY to get job_category
data2 = data
data2['job_category'] = data['JOB CATEGORY'].apply(normalize_position)
data2 = data2[data2.job_category.apply(lambda s: s != False)]
# data2['job_title'] = data['PRIMARY WORKING TITLE'].apply(normalize_position)
# data2 = data2[data2.job_title.apply(lambda s: s != False)]
data2

,INSTITUTION NAME,LAST NAME,FIRST NAME,INIT,AGE,INITIAL HIRE DATE,JOB CATEGORY,EMPLOYEE ANNUAL BASE SALARY,EMPLOYEE HOME DEPARTMENT,PRIMARY WORKING TITLE,job_category
0,ASU,Abbaraju,Pranith,NaN,34,"AUG 01, 2025",Assistant Professor,131500.0,Computer Information Systems,Assistant Professor,assistant_professor
2,ASU,Abu-Elzait,Sohad,NaN,45,"AUG 01, 2021",Assistant Professor,84020.0,Sustainable Technlgy & Built Envirn,Assistant Professor,assistant_professor
3,ASU,Acikgoz,Yalcin,NaN,42,"AUG 26, 2011",Associate Professor,86300.0,Psychology,Associate Professor,associate_professor
5,ASU,Adams,Elizabeth,A,37,"AUG 02, 2022",Assistant Professor,77030.0,Biology,Assistant Professor,assistant_professor
6,ASU,Adams,Kathleen,A,57,"JAN 01, 2004",Lecturer,48880.0,English,Senior Lecturer,lecturer
...,...,...,...,...,...,...,...,...,...,...,...
49223,WSSU,Xiong,Wen,NaN,57,"JUL 01, 2015",Professor,89922.0,World Languages and Culture,Professor,professor
49225,WSSU,Yi,John,T,59,"AUG 14, 2006",Professor,92123.0,Chemistry,Professor,professor
49228,WSSU,Zhang,Jinghua,NaN,50,"AUG 13, 2007",Associate Professor,102835.0,Computer Science,Professor,associate_professor
49229,WSSU,Zhang,Lei,NaN,59,"AUG 15, 2005",Associate Professor,90064.0,Chemistry,Professor,associate_professor


## Delete medical school faculty

It's common practice to treat medical school faculty separately in this type of analysis, as IPEDS does, for example. There are 968 unique academic departments spread across the UNC system. To identify the medical departments, we'll generate a text file with all 968 departments and ask ChatGPT to idenfity the ones within medical school.

Here's all the departments:

In [ ]:
with open('all_departments.txt', 'w') as dept_file:
    for dept in data2['EMPLOYEE HOME DEPARTMENT']:
        dept_file.write(f"{dept}\n")

I passed that `all_departments.txt` file to ChatGPT and it classified 109 of them as belonging to medical schools. I placed those into a file named `unc_medical_departments.txt`. Let's read that in and delete the medical departments:

In [13]:
with open('unc_medical_departments.txt', 'r') as med_dept_file:
    medical_depts = [line.strip() for line in med_dept_file.readlines()]
len(medical_depts)
data2 = data2[data2['EMPLOYEE HOME DEPARTMENT'].apply(lambda s: not s in medical_depts)]

## Polish

The data contains only school abbreviations; let's add full names:

In [16]:
def abbr_to_name(s):
    if s == 'ASU':
        return 'App State'
    elif s == 'ECSU':
        return 'Elizabeth City State'
    elif s == 'ECU':
        return 'Eastern Carolina'
    elif s == 'FSU':
        return 'Fayetteville State'
    elif s == 'NCA&T':
        return 'NC A&T'
    elif s == 'NCCU':
        return 'North Carolina Central'
    elif s == 'NCSU':
        return 'NC State'
    elif s == 'UNC-CH':
        return 'UNC Chapel Hill'
    elif s == 'UNCA':
        return 'UNC Asheville'
    elif s == 'UNCC':
        return 'UNC Charlotte'
    elif s == 'UNCG':
        return 'UNC Greensboro'
    elif s == 'UNCP':
        return 'UNC Pembroke'
    elif s == 'UNCW':
        return 'UNC Wilmington'
    elif s == 'WCU':
        return 'Western Carolina'
    elif s == 'WSSU':
        return 'Winston-Salem State'

data2['school_name'] = data['INSTITUTION NAME'].apply(abbr_to_name)
data2

/var/folders/dw/q0zlmbyn6qd55csnrk1ml4dh0000gp/T/ipykernel_32811/2656240368.py:33: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data2['school_name'] = data['INSTITUTION NAME'].apply(abbr_to_name)


,INSTITUTION NAME,LAST NAME,FIRST NAME,INIT,AGE,INITIAL HIRE DATE,JOB CATEGORY,EMPLOYEE ANNUAL BASE SALARY,EMPLOYEE HOME DEPARTMENT,PRIMARY WORKING TITLE,job_category,school_name
0,ASU,Abbaraju,Pranith,NaN,34,"AUG 01, 2025",Assistant Professor,131500.0,Computer Information Systems,Assistant Professor,assistant_professor,App State
2,ASU,Abu-Elzait,Sohad,NaN,45,"AUG 01, 2021",Assistant Professor,84020.0,Sustainable Technlgy & Built Envirn,Assistant Professor,assistant_professor,App State
3,ASU,Acikgoz,Yalcin,NaN,42,"AUG 26, 2011",Associate Professor,86300.0,Psychology,Associate Professor,associate_professor,App State
5,ASU,Adams,Elizabeth,A,37,"AUG 02, 2022",Assistant Professor,77030.0,Biology,Assistant Professor,assistant_professor,App State
6,ASU,Adams,Kathleen,A,57,"JAN 01, 2004",Lecturer,48880.0,English,Senior Lecturer,lecturer,App State
...,...,...,...,...,...,...,...,...,...,...,...,...
49223,WSSU,Xiong,Wen,NaN,57,"JUL 01, 2015",Professor,89922.0,World Languages and Culture,Professor,professor,Winston-Salem State
49225,WSSU,Yi,John,T,59,"AUG 14, 2006",Professor,92123.0,Chemistry,Professor,professor,Winston-Salem State
49228,WSSU,Zhang,Jinghua,NaN,50,"AUG 13, 2007",Associate Professor,102835.0,Computer Science,Professor,associate_professor,Winston-Salem State
49229,WSSU,Zhang,Lei,NaN,59,"AUG 15, 2005",Associate Professor,90064.0,Chemistry,Professor,associate_professor,Winston-Salem State


Here's the final data:

In [28]:
final_data = data2.rename(columns = {
    'INSTITUTION NAME': 'school_abbr',
    'EMPLOYEE HOME DEPARTMENT': 'dept',
    'EMPLOYEE ANNUAL BASE SALARY': 'salary'
})[['school_abbr', 'school_name', 'job_category', 'salary']]
final_data

,school_abbr,school_name,job_category,salary
0,ASU,App State,assistant_professor,131500.0
2,ASU,App State,assistant_professor,84020.0
3,ASU,App State,associate_professor,86300.0
5,ASU,App State,assistant_professor,77030.0
6,ASU,App State,lecturer,48880.0
...,...,...,...,...
49223,WSSU,Winston-Salem State,professor,89922.0
49225,WSSU,Winston-Salem State,professor,92123.0
49228,WSSU,Winston-Salem State,associate_professor,102835.0
49229,WSSU,Winston-Salem State,associate_professor,90064.0


It would be easy to include the `dept` field but I don't need it and it increases the file size by 50%.

In [29]:
final_data.to_csv('unc_system_condensed_salaries.csv', index=False)